# Manage Users DB

In [ ]:
import pandas as pd
import sys
import os

# Adding project's root folder to sys.path
sys.path.append(os.path.abspath("../.."))

from domain.users.identification import find_user_by_prompt
from infrastructure.crypto.fernet import decode_symm_crypt_key
from infrastructure.persistence.users.repository import (
    delete_user_by_field,
    get_user_by_field,
    init_db,
    insert_user_list,
)

## Initial Setup

Option A. Introduce these required fields:

In [ ]:
real_name = "MyRealName"
access_name = "MyAccessName"  # Identification for Jarvis to recognize the user
jarvis_name = "MyJarvisName" # Jarvis's name for the user"
is_female = True # If the user identifies as a woman
admin = False # If the user has admin privileges

user_list = [
    {
        "real_name": real_name,
        "access_name": access_name,
        "jarvis_name": jarvis_name,
        "is_female": is_female,
        "admin": admin
    }
]

user_list

Option B. Use Users Info CSV (Secret)

In [ ]:
# Loading CSV
df = pd.read_csv("secret_users_info.csv")

# Converting boolean values from text ("true"/"false")
df['is_female'] = df['is_female'].astype(str).str.lower() == 'true'
df['admin'] = df['admin'].astype(str).str.lower() == 'true'

# Convertir a lista de diccionarios
user_list = df.to_dict(orient='records')
user_list

In [ ]:
# Initialize database - Execute if DB does not exist yet
init_db()

## Adding user to the DB

In [ ]:
insert_user_list(user_list)

### Testing user is correctly added

In [ ]:
user = get_user_by_field(
    field="access_name",
    value=access_name,
    is_sensitive=True
)

In [ ]:
decode_symm_crypt_key(user.get("password"))

In [ ]:
prompt = f"Hola, soy {access_name}, quiero hablar contigo"
name = find_user_by_prompt(prompt=prompt)["jarvis_name"]
print(name) # Expected output: MyJarvisName

## Deleting user by username

In [ ]:
# Clean up the database by removing the user
delete_user_by_field(field="access_name", value=access_name, is_sensitive=True)

### Testing user is correctly deleted

In [ ]:
prompt = f"Hola, soy {access_name}, quiero hablar contigo"
try:
    name = find_user_by_prompt(prompt=prompt)["jarvis_name"]
except TypeError:
    print("User not found")  # Expected output: User not found